# Validity-Gated CCR for Korean Hate Speech Detection
**4 ablation conditions × 3 seeds × 3 epochs on K-HATERS (SUBSET=2000)**

In [ ]:
# 패키지 설치
!pip install -q transformers datasets kiwipiepy scikit-learn scipy

In [ ]:

import os, json, random, gc
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score
from scipy import stats
from tqdm.notebook import tqdm
from datasets import load_dataset
from kiwipiepy import Kiwi
import warnings; warnings.filterwarnings('ignore')

# ── Config ────────────────────────────────────────────────────────────────────
BASE_DIR    = '/content'
MODEL_NAME  = 'klue/roberta-base'
MAX_LEN     = 128
BATCH_SIZE  = 32
EPOCHS      = 3
LR          = 2e-5
WEIGHT_DECAY= 0.01
LAMBDA      = 0.1
SEEDS       = [42, 123, 456]
SUBSET      = 0       # 0 = full 172K

CKPT_DIR    = os.path.join(BASE_DIR, 'checkpoints')
RESULT_PATH = os.path.join(BASE_DIR, 'results.json')
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'Model  : {MODEL_NAME}')
print(f'SUBSET : {"full" if SUBSET == 0 else SUBSET}')


In [ ]:

# ── Seed ─────────────────────────────────────────────────────────────────────
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ── Identity swap pairs ───────────────────────────────────────────────────────
SWAP_PAIRS_BY_CAT = {
    'gender':     [('여성', '남성'), ('여자', '남자'), ('여성들', '남성들'),
                   ('여자들', '남자들'), ('페미니스트', '남성우월주의자'),
                   ('페미', '한남'), ('메갈', '한남'), ('한녀', '한남')],
    'religion':   [('무슬림', '기독교인'), ('이슬람', '기독교'),
                   ('무슬림', '천주교인'), ('이슬람교도', '기독교인')],
    'ethnicity':  [('조선족', '한국인'), ('외국인', '내국인'),
                   ('탈북민', '남한사람'),
                   ('베트남인', '한국인'), ('일본인', '한국인'),
                   ('재일교포', '한국인'), ('동남아인', '한국인')],
    'age':        [('노인', '청년'), ('노년층', '청년층'),
                   ('할머니', '젊은여자'), ('할아버지', '젊은남자')],
    'sexuality':  [('동성애자', '이성애자'), ('게이', '이성애자'),
                   ('레즈비언', '이성애자'), ('성소수자', '이성애자'),
                   ('트랜스젠더', '이성애자'), ('퀴어', '이성애자')],
    'disability': [('장애인', '비장애인'), ('정신장애인', '비장애인'),
                   ('지적장애인', '비장애인')],
}

# term → (counterpart, category) — directed only (a→b), no reverse to avoid overwrite
SWAP_MAP: dict = {}
for cat, pairs in SWAP_PAIRS_BY_CAT.items():
    for a, b in pairs:
        SWAP_MAP[a] = (b, cat)
SWAP_KEYS = sorted(SWAP_MAP.keys(), key=len, reverse=True)
print(f'Swap terms: {len(SWAP_KEYS)}개 ({len(SWAP_PAIRS_BY_CAT)} categories)')

kiwi = Kiwi()

# ── Swap functions ────────────────────────────────────────────────────────────
def find_swap(text: str):
    """Validity-Gated: 형태소 경계 + 단일 term + 동일 카테고리"""
    tokens = kiwi.tokenize(text)
    token_forms = [t.form for t in tokens]
    found = [term for term in SWAP_KEYS if term in token_forms]
    if len(set(found)) >= 2:
        return None, None, None
    if found:
        term = found[0]
        counterpart, cat = SWAP_MAP[term]
        return term, counterpart, cat
    return None, None, None

def find_swap_naive(text: str):
    """Naive Swap: 단순 substring 매칭"""
    for term in SWAP_KEYS:
        if term in text:
            counterpart, cat = SWAP_MAP[term]
            return term, counterpart, cat
    return None, None, None

def _has_batchim(char: str) -> bool:
    code = ord(char)
    if not (0xAC00 <= code <= 0xD7A3):
        return False
    return (code - 0xAC00) % 28 != 0

def _ends_with_rieul(char: str) -> bool:
    code = ord(char)
    if not (0xAC00 <= code <= 0xD7A3):
        return False
    return (code - 0xAC00) % 28 == 8

_JOSA_VOWEL = {
    '이': '가', '을': '를', '은': '는',
    '과': '와', '아': '야',
    '이나': '나', '이랑': '랑', '이든': '든',
    '이라고': '라고', '이라며': '라며', '이라는': '라는',
    '이라서': '라서', '이라도': '라도', '이라면': '라면',
    '이란': '란', '이야': '야',
}
_JOSA_CONS = {v: k for k, v in _JOSA_VOWEL.items()}
_ALT_JOSA  = set(_JOSA_VOWEL) | set(_JOSA_CONS) | {'으로', '로'}

def _adjust_josa(new_term: str, josa: str) -> str:
    if not new_term:
        return josa
    last = new_term[-1]
    if josa in ('으로', '로'):
        return '으로' if (_has_batchim(last) and not _ends_with_rieul(last)) else '로'
    if _has_batchim(last):
        return _JOSA_CONS.get(josa, josa)
    return _JOSA_VOWEL.get(josa, josa)

def _check_grammar(cf: str, swap_term: str) -> bool:
    """swap_term 뒤에 잘못된 조사 패턴이 있는지 확인."""
    if not swap_term:
        return True
    last = swap_term[-1]
    has_batchim = _has_batchim(last)
    if has_batchim:
        bad = [swap_term + j for j in _JOSA_VOWEL.values()]
    else:
        bad = [swap_term + j for j in _JOSA_VOWEL.keys()]
    return not any(p in cf for p in bad)

def make_swap(text: str, orig_term: str, new_term: str) -> str:
    """형태소 경계 기반 교체 + 뒤따르는 교체형 조사 자동 조정."""
    tokens = kiwi.tokenize(text)
    subs: list[tuple[int, int, str]] = []
    for i, t in enumerate(tokens):
        if t.form == orig_term:
            subs.append((t.start, t.start + t.len, new_term))
            if i + 1 < len(tokens):
                nxt = tokens[i + 1]
                if str(nxt.tag).startswith('J') and nxt.form in _ALT_JOSA:
                    fixed = _adjust_josa(new_term, nxt.form)
                    if fixed != nxt.form:
                        subs.append((nxt.start, nxt.start + nxt.len, fixed))
    if not subs:
        return text
    for start, end, repl in sorted(subs, key=lambda x: x[0], reverse=True):
        text = text[:start] + repl + text[end:]
    return text

def make_swap_naive(text: str, orig_term: str, new_term: str) -> str:
    return text.replace(orig_term, new_term)

# ── Validity criteria ─────────────────────────────────────────────────────────
_SEMANTIC_BLACKLIST = {
    'ethnicity': ['입국', '방역', '국경', '불법체류', '이민', '귀화', '출입국',
                  '난민', '밀입국', '추방', '체류'],
    'gender':    ['위안부', '임신', '출산', '생식', '여학생', '남학생', '자궁',
                  '성폭력', '성폭행', '성매매', '강간', '몰카'],
    'religion':  ['부르카', '히잡', '테러', '지하드', '성전', '탈레반', '샤리아'],
    'age':       ['위안부', '전쟁', '일제', '역사적'],
    'sexuality': ['결혼', '입양', '헌혈', '군대', '병역'],
}

_ASYMMETRIC_PAIRS = {
    ('게이', '이성애자'), ('이성애자', '게이'),
    ('레즈비언', '이성애자'), ('이성애자', '레즈비언'),
    ('트랜스젠더', '이성애자'), ('이성애자', '트랜스젠더'),
    ('성소수자', '이성애자'), ('이성애자', '성소수자'),
    ('퀴어', '이성애자'), ('이성애자', '퀴어'),
    ('할머니', '젊은여자'), ('젊은여자', '할머니'),
    ('할아버지', '젊은남자'), ('젊은남자', '할아버지'),
}

def compute_validity(original: str, cf: str, orig_term: str, swap_term: str, cat: str) -> dict:
    """4-criterion validity gate."""
    same_category = True
    valid_grammar = _check_grammar(cf, swap_term)
    text_for_check = original + " " + cf
    blacklist = _SEMANTIC_BLACKLIST.get(cat, [])
    valid_semantics = not any(kw in text_for_check for kw in blacklist)
    label_preserving = (
        (orig_term, swap_term) not in _ASYMMETRIC_PAIRS
        and valid_semantics
    )
    use_for_ccr = same_category and valid_grammar and valid_semantics and label_preserving
    return {
        'same_category': same_category,
        'valid_grammar': valid_grammar,
        'valid_semantics': valid_semantics,
        'label_preserving': label_preserving,
        'use_for_ccr': use_for_ccr,
    }


In [ ]:

# ── K-HATERS 로딩 ─────────────────────────────────────────────────────────────
HATE_LABELS = {'offensive', 'L1_hate', 'L2_hate'}

def to_binary(label: str) -> int:
    return 1 if label in HATE_LABELS else 0

def load_khaters(split: str, subset: int = 0, seed: int = 42):
    ds = load_dataset('humane-lab/K-HATERS', split=split)
    examples = []
    for row in ds:
        text = row['text'].strip()
        if not text: continue
        label = to_binary(row['label'])
        targets = row.get('target_label') or []
        examples.append((text, label, targets))
    if subset:
        rng = random.Random(seed)
        pos = [e for e in examples if e[1] == 1]
        neg = [e for e in examples if e[1] == 0]
        rng.shuffle(pos); rng.shuffle(neg)
        examples = pos[:subset // 2] + neg[:subset // 2]
        rng.shuffle(examples)
    return examples

def save_cf_pairs(examples, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    pairs = []
    for text, label, targets in examples:
        orig_term, swap_term, cat = find_swap(text)
        if orig_term is None: continue
        cf_text  = make_swap(text, orig_term, swap_term)
        validity = compute_validity(text, cf_text, orig_term, swap_term, cat)
        pairs.append({'original': text, 'cf': cf_text, 'orig_term': orig_term,
                      'swap_term': swap_term, 'category': cat,
                      'label': label, 'targets': targets, **validity})
    with open(path, 'w', encoding='utf-8') as f:
        for p in pairs:
            f.write(json.dumps(p, ensure_ascii=False) + '\n')
    n_valid = sum(1 for p in pairs if p['use_for_ccr'])
    print(f'CF pairs saved → {path}  (total={len(pairs)}, use_for_ccr={n_valid})')

print('Loading K-HATERS...')
raw_train = load_khaters('train',      SUBSET)
raw_val   = load_khaters('validation', 0)
raw_test  = load_khaters('test',       0)

hate_rate = sum(l for _, l, _ in raw_train) / len(raw_train)
print(f'train={len(raw_train)}  val={len(raw_val)}  test={len(raw_test)}')
print(f'train hate rate: {hate_rate:.3f}')

n_swap = sum(1 for t, _, _ in raw_train if find_swap(t)[0] is not None)
print(f'swappable: {n_swap} / {len(raw_train)} ({100*n_swap/len(raw_train):.1f}%)')

save_cf_pairs(raw_train, '/content/cf_pairs_train.jsonl')


In [ ]:

# ── Dataset & Model ───────────────────────────────────────────────────────────
class HatersDataset(Dataset):
    def __init__(self, examples, tokenizer, max_len, mode='none'):
        assert mode in ('none', 'mask', 'swap', 'gated')
        self.tok, self.max_len, self.mode = tokenizer, max_len, mode
        self.items = []
        for text, label, targets in examples:
            if mode == 'swap':
                orig_term, swap_term, cat = find_swap_naive(text)
            else:
                orig_term, swap_term, cat = find_swap(text)
            has_swap = orig_term is not None
            cf_text, cf_valid = None, False

            if has_swap and mode == 'swap':
                cf_text  = make_swap_naive(text, orig_term, swap_term)
                cf_valid = True   # no validity check (naive)
            elif has_swap and mode == 'gated':
                cf_text  = make_swap(text, orig_term, swap_term)
                validity = compute_validity(text, cf_text, orig_term, swap_term, cat)
                cf_valid = validity['use_for_ccr']   # 4-criterion gate
            elif mode == 'mask' and has_swap:
                cf_text  = make_swap(text, orig_term, tokenizer.mask_token)
                cf_valid = True

            self.items.append({'text': text, 'label': label, 'targets': targets,
                               'cf_text': cf_text, 'cf_valid': cf_valid})

    def _enc(self, text):
        e = self.tok(text, max_length=self.max_len, padding='max_length',
                     truncation=True, return_tensors='pt')
        return e['input_ids'].squeeze(0), e['attention_mask'].squeeze(0)

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        it = self.items[idx]
        ids, mask = self._enc(it['text'])
        cf_src = it['cf_text'] if it['cf_text'] is not None else it['text']
        cf_ids, cf_mask = self._enc(cf_src)
        return {'input_ids': ids, 'attention_mask': mask,
                'label': torch.tensor(it['label'], dtype=torch.long),
                'cf_valid': torch.tensor(it['cf_valid'], dtype=torch.bool),
                'cf_input_ids': cf_ids, 'cf_attention_mask': cf_mask}


class HateDetector(nn.Module):
    def __init__(self, model_name, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, 2)

    def forward(self, input_ids, attention_mask):
        cls = self.encoder(input_ids=input_ids,
                           attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.classifier(self.dropout(cls))

print('Dataset & Model classes defined.')


In [ ]:

# ── Loss & Train/Eval ─────────────────────────────────────────────────────────
def sym_kl(p, q):
    return (F.kl_div(q.log(), p, reduction='batchmean') +
            F.kl_div(p.log(), q, reduction='batchmean')) / 2

def train_epoch(model, loader, optimizer, lam, use_cons):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc='train', leave=False):
        ids      = batch['input_ids'].to(device)
        mask     = batch['attention_mask'].to(device)
        labels   = batch['label'].to(device)
        cf_valid = batch['cf_valid'].to(device)

        logits = model(ids, mask)
        loss = F.cross_entropy(logits, labels)

        if use_cons and cf_valid.any():
            cf_ids  = batch['cf_input_ids'].to(device)
            cf_mask = batch['cf_attention_mask'].to(device)
            p = F.softmax(logits[cf_valid], dim=-1)
            q = F.softmax(model(cf_ids[cf_valid], cf_mask[cf_valid]), dim=-1)
            loss = loss + lam * sym_kl(p, q)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    preds, labels = [], []
    for batch in tqdm(loader, desc='eval', leave=False):
        logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        preds.extend(logits.argmax(-1).cpu().tolist())
        labels.extend(batch['label'].tolist())
    return f1_score(labels, preds, average='macro')

@torch.no_grad()
def eval_fairness(model, examples, tokenizer):
    """
    Flip Rate / Logit Gap: swappable 예제 (find_swap 기준) 대상
    Per-group FPR: lexicon 기반 category 할당 (target_label 미사용)
      - K-HATERS target_label은 label=1에만 존재 → label=0 FPR 계산 불가
      - 대신 문장 내 identity term 카테고리로 그룹 분류
    """
    model.eval()
    flip_count, total_swap = 0, 0
    logit_gaps = []
    group_fp = defaultdict(int)
    group_tn = defaultdict(int)

    for text, label, _ in tqdm(examples, desc='fairness', leave=False):
        enc = tokenizer(text, max_length=MAX_LEN, padding='max_length',
                        truncation=True, return_tensors='pt')
        logits = model(enc['input_ids'].to(device), enc['attention_mask'].to(device))
        prob  = F.softmax(logits, dim=-1)[0, 1].item()
        pred  = int(prob >= 0.5)

        orig_term, swap_term, cat = find_swap(text)
        cf_pred, cf_prob = None, None
        if orig_term:
            cf_text = make_swap(text, orig_term, swap_term)
            enc2 = tokenizer(cf_text, max_length=MAX_LEN, padding='max_length',
                             truncation=True, return_tensors='pt')
            logits2 = model(enc2['input_ids'].to(device), enc2['attention_mask'].to(device))
            cf_prob = F.softmax(logits2, dim=-1)[0, 1].item()
            cf_pred = int(cf_prob >= 0.5)
            total_swap += 1
            if pred != cf_pred:
                flip_count += 1
            logit_gaps.append(abs(prob - cf_prob))

        # FPR 계산: lexicon 기반 category로 그룹 분류
        if label == 0:
            grp = cat if cat else 'none'
            if pred == 1:
                group_fp[grp] += 1
            else:
                group_tn[grp] += 1

    flip_rate = flip_count / total_swap if total_swap else 0.0
    mean_gap  = float(np.mean(logit_gaps)) if logit_gaps else 0.0

    per_group_fpr = {}
    for grp in set(list(group_fp.keys()) + list(group_tn.keys())):
        denom = group_fp[grp] + group_tn[grp]
        per_group_fpr[grp] = group_fp[grp] / denom if denom else 0.0

    # FPR gap: identity 그룹들 사이의 최대 격차 ('none' 제외)
    identity_fprs = {k: v for k, v in per_group_fpr.items() if k != 'none'}
    fpr_vals = list(identity_fprs.values())
    fpr_gap  = (max(fpr_vals) - min(fpr_vals)) if len(fpr_vals) >= 2 else 0.0

    return {'flip_rate': flip_rate, 'logit_gap': mean_gap,
            'fpr_gap': fpr_gap, 'per_group_fpr': per_group_fpr}

print('Training functions defined.')


In [ ]:

# ── Main Experiment ───────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

ABLATIONS = [
    dict(tag='Baseline',       mode='none',  use_cons=False, lam=0.0),
    dict(tag='Validity-Gated', mode='gated', use_cons=True,  lam=LAMBDA),
    # dict(tag='Masking CCR',    mode='mask',  use_cons=True,  lam=LAMBDA),
    # dict(tag='Naive Swap',     mode='swap',  use_cons=True,  lam=LAMBDA),
]

all_results = []

for abl in ABLATIONS:
    tag, mode, use_cons, lam = abl['tag'], abl['mode'], abl['use_cons'], abl['lam']
    print(f'\n{"#"*60}')
    print(f'  Experiment: {tag}')
    print(f'{"#"*60}')

    seed_metrics = []
    for seed in SEEDS:
        print(f'\n[{tag}] seed={seed}')
        set_seed(seed)

        train_ds = HatersDataset(raw_train, tokenizer, MAX_LEN, mode)
        val_ds   = HatersDataset(raw_val,   tokenizer, MAX_LEN, 'none')
        test_ds  = HatersDataset(raw_test,  tokenizer, MAX_LEN, 'none')
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
        test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

        model = HateDetector(MODEL_NAME).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_f1, best_state = 0, None
        for epoch in range(1, EPOCHS + 1):
            tr_loss = train_epoch(model, train_loader, optimizer, lam, use_cons)
            val_f1  = eval_model(model, val_loader)
            print(f'  epoch {epoch}/{EPOCHS}  loss={tr_loss:.4f}  val_f1={val_f1:.4f}')
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        test_f1  = eval_model(model, test_loader)
        fairness = eval_fairness(model, raw_test, tokenizer)

        print(f'  Test F1={test_f1:.4f}  flip={fairness["flip_rate"]:.4f}  '
              f'gap={fairness["logit_gap"]:.4f}  fpr_gap={fairness["fpr_gap"]:.4f}')
        print('  per-group FPR: ' +
              '  '.join(f'{k}={v:.3f}' for k, v in sorted(fairness['per_group_fpr'].items())))

        seed_metrics.append({'seed': seed, 'test_f1': test_f1, **fairness})
        del model; gc.collect(); torch.cuda.empty_cache()

    all_results.append({'tag': tag, 'mode': mode, 'lam': lam, 'seeds': seed_metrics})

with open(RESULT_PATH, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
print(f'\nResults saved → {RESULT_PATH}')


In [ ]:

# ── 결과 요약 테이블 + 파일 다운로드 ──────────────────────────────────────────
import numpy as np
from google.colab import files

with open(RESULT_PATH, encoding='utf-8') as f:
    results = json.load(f)

print(f"{'Condition':<22} {'F1':>6} {'FlipRate':>9} {'LogitGap':>9} {'FPR Gap':>8}")
print('-' * 58)
for r in results:
    ms = r['seeds']
    f1   = np.mean([m['test_f1']   for m in ms])
    flip = np.mean([m['flip_rate'] for m in ms])
    gap  = np.mean([m['logit_gap'] for m in ms])
    fpr  = np.mean([m['fpr_gap']   for m in ms])
    print(f'{r["tag"]:<22} {f1:>6.4f} {flip:>9.4f} {gap:>9.4f} {fpr:>8.4f}')

# 파일 다운로드
files.download(RESULT_PATH)
files.download('/content/cf_pairs_train.jsonl')
